# UPI Fraud Rings on Colab (Team Quadruples)
Before running: put **`upi-fraud-rings-gradient-wars.zip`** (the competition data) and **`colab_code.zip`** (made by `python colab_bundle.py`) in a Google Drive folder named **`upi-fraud-rings`** (`My Drive/upi-fraud-rings/`). Then **Runtime → Change runtime type → T4 GPU**, and **Runtime → Run all**.

Predictions are copied to `My Drive/upi-fraud-rings/colab_preds/` after every job. If Colab disconnects, **Run all** again: finished jobs are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/upi-fraud-rings'
import os
for f in ('upi-fraud-rings-gradient-wars.zip', 'colab_code.zip'):
    assert os.path.exists(f'{DRIVE}/{f}'), f'{f} not found in My Drive/upi-fraud-rings'

In [ ]:
os.makedirs('/content/upi/data', exist_ok=True)
%cd /content/upi
!unzip -q -o "{DRIVE}/colab_code.zip" -d /content/upi
!unzip -q -o -j "{DRIVE}/upi-fraud-rings-gradient-wars.zip" -d /content/upi/data
!pip -q install lightgbm xgboost catboost
!mkdir -p preds models logs submissions
!ls -la /content/upi /content/upi/data

In [ ]:
# feature blocks (built over train + test together), then the folds if the zip had none
FEATS = 'row'
for b in FEATS.split('+'):
    if not os.path.exists(f'data/feat_{b}.parquet'):
        !python features.py {b}
if not os.path.exists('folds.csv'):
    !python make_folds.py

In [ ]:
# one line per job: (tag, train_gbm.py arguments). Finished jobs (already on Drive) are skipped.
import glob, shutil, time
OUT = f'{DRIVE}/colab_preds'; os.makedirs(OUT, exist_ok=True)
GPU_XGB = '''--params '{"device": "cuda"}' '''
GPU_CAT = '''--params '{"task_type": "GPU"}' '''
JOBS = [
    ('xgb',  f'--model xgb --balanced --cap 150000 --seeds 3 --feats {FEATS} {GPU_XGB}'),
    ('cat',  f'--model cat --balanced --cap 150000 --seeds 3 --feats {FEATS} {GPU_CAT}'),
    ('lgb5', f'--model lgb --balanced --cap 150000 --seeds 5 --feats {FEATS}'),
]
for tag, args in JOBS:
    if os.path.exists(f'{OUT}/{tag}_test.npy'):
        print(f'{tag}: already on Drive, skipping'); continue
    t0 = time.time()
    !python train_gbm.py --tag {tag} {args}
    for p in glob.glob(f'preds/{tag}_*.npy'): shutil.copy(p, OUT)
    shutil.copy('logs/scores.log', OUT)
    print(f'{tag} done in {(time.time() - t0) / 60:.1f} min, saved to Drive')

In [ ]:
!ls -la "{DRIVE}/colab_preds"
!cat logs/scores.log